# 📊 ขั้นตอนที่ 2: Data Acquisition & Understanding
### (การรวบรวมและทำความเข้าใจข้อมูล)
---
**วัตถุประสงค์ในขั้นตอนนี้:**
1. รวบรวมข้อมูลประวัติการแข่งขันฟุตบอลและราคาเปิดจากฐานข้อมูล
2. สำรวจโครงสร้างข้อมูล ชนิดของตัวแปร และตรวจสอบค่าสูญหาย (Missing Values)
3. วิเคราะห์สถิติเบื้องต้น (EDA) เช่น อัตราการยิงประตู, ความได้เปรียบของการเล่นในบ้าน และค่าน้ำของเจ้ามือ

## 1. นำเข้าไลบรารีที่จำเป็น (Imports)

In [ ]:
import os, sys
# จัดการ Working Directory ให้ชี้ไปที่โฟลเดอร์หลักของโปรเจกต์เสมอ
if not os.path.exists('data') and os.path.exists('../data'):
    os.chdir('..')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
import sqlite3
import pandas as pd
import numpy as np

# กำหนดการแสดงผลของ pandas
pd.set_option('display.max_columns', 20)
print("✅ นำเข้าไลบรารีสำเร็จ")

## 2. รวบรวมข้อมูลจากฐานข้อมูล SQLite (`data/football.db`)

In [ ]:
con = sqlite3.connect('data/football.db')

query = '''
SELECT match_id, provider, league, season, kickoff_at, home_team, away_team,
       home_goals, away_goals, fav_side, ah_open, ou_line_open,
       ah_fav_price, ah_dog_price, ou_over_price, ou_under_price
FROM matches
WHERE home_goals IS NOT NULL AND ah_fav_price IS NOT NULL
ORDER BY kickoff_at ASC
'''
df = pd.read_sql_query(query, con)
con.close()

print(f"จำนวนนัดที่ดึงมาได้: {len(df):,} นัด")
df.head()

## 3. สำรวจโครงสร้างของข้อมูล (Data Structure & Data Types)

In [ ]:
print("ขนาดตารางข้อมูล (Rows, Columns):", df.shape)
print("\nชนิดข้อมูลแต่ละคอลัมน์:")
print(df.dtypes)

## 4. ตรวจสอบค่าสูญหาย (Missing Values Analysis)

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
df_missing = pd.DataFrame({'Missing Count': missing, 'Percentage (%)': missing_pct.round(2)})
df_missing

## 5. การวิเคราะห์เชิงสถิติ (Descriptive Statistics & Exploration)

In [ ]:
df['total_goals'] = df['home_goals'] + df['away_goals']

print("=== สถิติการทำประตูต่อนัด ===")
print(f"ประตูเฉลี่ยทีมเจ้าบ้าน: {df['home_goals'].mean():.2f} ลูก")
print(f"ประตูเฉลี่ยทีมเยือน:   {df['away_goals'].mean():.2f} ลูก")
print(f"ประตูรวมเฉลี่ยต่อนัด:    {df['total_goals'].mean():.2f} ลูก")

print("\n=== จำนวนนัดแยกตามลีก ===")
print(df['league'].value_counts())

## 6. สำรวจค่าน้ำของเจ้ามือ (Bookmaker Margin / Overround)

In [ ]:
# คำนวณผลรวมความน่าจะเป็นก่อนตัดค่าน้ำ (ปกติจะเกิน 1.0 หรือ 100% เสมอ)
df['ah_margin'] = (1.0 / df['ah_fav_price']) + (1.0 / df['ah_dog_price'])
df['ou_margin'] = (1.0 / df['ou_over_price']) + (1.0 / df['ou_under_price'])

print(f"ค่าน้ำเฉลี่ยตลาดแฮนดิแคป (AH Overround): {(df['ah_margin'].mean() - 1.0) * 100:.2f}%")
print(f"ค่าน้ำเฉลี่ยตลาดสูงต่ำ (OU Overround):    {(df['ou_margin'].mean() - 1.0) * 100:.2f}%")

print("\n💡 สรุปขั้นตอนที่ 2: ข้อมูลมีความสมบูรณ์ และเห็นได้ชัดเจนว่าโต๊ะมีการหักค่าน้ำเฉลี่ยประมาณ 5-6%")